# TwinLens: reproduce the headline in 5 minutes

TwinLens is a digital twin of a transformer's internals that checks how far its own attribution-graph explanations can be trusted. This notebook recomputes the headline from the saved edit sweeps (no GPU, no model downloads): the frozen graph's predicted effect of an edit versus the real effect, and the same after the Jacobian fix, on GPT-2 small and Qwen3-0.6B.

Full write-up: [report](https://spiliosdimakopoulos.github.io/twinlens/report/) | DOI: [10.5281/zenodo.23437556](https://doi.org/10.5281/zenodo.23437556)

In [ ]:
import os
if os.path.exists("../experiments/results"):  # running from notebooks/ inside the repo
    os.chdir("..")
# Colab: clone the repo. Locally (already inside the repo): skip.
if not os.path.exists("experiments/results"):
    if not os.path.exists("twinlens"):
        !git clone --depth 1 https://github.com/SpiliosDimakopoulos/twinlens
    %cd twinlens
    %pip install -q pandas pyarrow scipy matplotlib
print(os.getcwd())

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

R = "experiments/results/"
GRAPH, FIX = "linear", "linear_fulljac"   # frozen graph vs. Jacobian fix

def load(name):
    df = pd.read_parquet(R + name)
    for lv in (GRAPH, FIX):
        err = (df[f"{lv}_dlogp"] - df.real_dlogp).abs()
        df[f"{lv}_err"] = err
        df[f"{lv}_catastrophic"] = err > 1.0          # miss by more than 1 nat
    df["real_big"] = df.real_dlogp.abs() > 0.05
    return df

def metrics(d, lv):
    big = d[d.real_big]
    return {
        "spearman": d[[f"{lv}_dlogp", "real_dlogp"]].corr(method="spearman").iloc[0, 1],
        "sign_agree": ((big[f"{lv}_dlogp"] > 0) == (big.real_dlogp > 0)).mean(),
        "catastrophic": d[f"{lv}_catastrophic"].mean(),
    }

data = {"GPT-2 small": load("fidelity.parquet"), "Qwen3-0.6B": load("fidelity_qwen3.parquet")}

rows = []
for model, df in data.items():
    for name, lv in (("frozen graph", GRAPH), ("fix", FIX)):
        m = metrics(df, lv)
        rows.append({"model": model, "twin": name, "n_edits": len(df),
                     "spearman": round(m["spearman"], 3),
                     "sign_agree": f"{m['sign_agree']:.1%}",
                     "catastrophic >1 nat": f"{m['catastrophic']:.2%}"})
pd.DataFrame(rows).set_index(["model", "twin"])

In [ ]:
rng = np.random.default_rng(0)

def boot_diff(df, n=1000):
    groups = {k: g for k, g in df.groupby("prompt_id")}
    ids = list(groups)
    diff = lambda d: metrics(d, FIX)["spearman"] - metrics(d, GRAPH)["spearman"]
    samples = [diff(pd.concat([groups[i] for i in rng.choice(ids, len(ids))])) for _ in range(n)]
    lo, hi = np.percentile(samples, [2.5, 97.5])
    return diff(df), lo, hi

for model, df in data.items():
    d, lo, hi = boot_diff(df)
    print(f"{model}: paired delta-rho = {d:+.3f}  (95% prompt-cluster bootstrap CI {lo:+.3f}, {hi:+.3f})")

In [ ]:
df = data["GPT-2 small"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
lim = np.percentile(np.abs(df.real_dlogp), 99)
for ax, (name, lv) in zip(axes, (("Frozen graph", GRAPH), ("Graph + Jacobian fix", FIX))):
    ax.scatter(df.real_dlogp, df[f"{lv}_dlogp"], s=4, alpha=0.25)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    rho = metrics(df, lv)["spearman"]
    ax.set_title(f"{name}  (Spearman rho = {rho:.2f})")
    ax.set_xlabel("real effect (change in log-prob)")
axes[0].set_ylabel("predicted effect")
plt.tight_layout(); plt.show()

## Conclusion

The frozen attribution graph ranks edit effects poorly (GPT-2 rho about 0.20) and sometimes misses by more than a nat. Adding the LayerNorm and attention Jacobians lifts GPT-2 to rho about 0.53 and cuts catastrophic misses from 2.2% to 0.05%. The same fix transfers to Qwen3-0.6B (rho about 0.34 to 0.45), with paired intervals that exclude zero. See the report for the trust model and ablations.